In [8]:

import requests

url = "https://data.cityofchicago.org/resource/68nd-jvt3.csv"

params = {
    "$where": (
        "crash_date >= '2025-01-01T00:00:00' "
        "AND crash_date < '2027-01-01T00:00:00'"
    ),
    "$limit": 5000000
}

output_file = "/content/Traffic_Crashes_Vehicles_2025_2026.csv"

with requests.get(
    url, params=params, stream=True, timeout=180
) as response:
    response.raise_for_status()

    with open(output_file, "wb") as f:
        for chunk in response.iter_content(
            chunk_size=1024 * 1024
        ):
            if chunk:
                f.write(chunk)

print("Download complete!")


Download complete!


In [9]:

import pandas as pd
import os

file_path = "/content/Traffic_Crashes_Vehicles_2025_2026.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Dataset shape:", df.shape)
print("File size (MB):",
      round(os.path.getsize(file_path) / 1024**2, 2))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


Dataset shape: (418477, 71)
File size (MB): 143.47

Columns:
['crash_unit_id', 'crash_record_id', 'crash_date', 'unit_no', 'unit_type', 'num_passengers', 'vehicle_id', 'cmrc_veh_i', 'make', 'model', 'lic_plate_state', 'vehicle_year', 'vehicle_defect', 'vehicle_type', 'vehicle_use', 'travel_direction', 'maneuver', 'towed_i', 'fire_i', 'occupant_cnt', 'exceed_speed_limit_i', 'towed_by', 'towed_to', 'area_00_i', 'area_01_i', 'area_02_i', 'area_03_i', 'area_04_i', 'area_05_i', 'area_06_i', 'area_07_i', 'area_08_i', 'area_09_i', 'area_10_i', 'area_11_i', 'area_12_i', 'area_99_i', 'first_contact_point', 'cmv_id', 'usdot_no', 'ccmc_no', 'ilcc_no', 'commercial_src', 'gvwr', 'carrier_name', 'carrier_state', 'carrier_city', 'hazmat_placards_i', 'hazmat_name', 'un_no', 'hazmat_present_i', 'hazmat_report_i', 'hazmat_report_no', 'mcs_report_i', 'mcs_report_no', 'hazmat_vio_cause_crash_i', 'mcs_vio_cause_crash_i', 'idot_permit_no', 'wide_load_i', 'trailer1_width', 'trailer2_width', 'trailer1_length'

,crash_unit_id,crash_record_id,crash_date,unit_no,unit_type,num_passengers,vehicle_id,cmrc_veh_i,make,model,...,trailer1_length,trailer2_length,total_vehicle_length,axle_cnt,vehicle_config,cargo_body_type,load_type,hazmat_out_of_service_i,mcs_out_of_service_i,hazmat_class
0,2430972,cf7c14be28924d4de48bd41c392260cddc55c3c327f3bb...,2026-11-10T21:38:00.000,1,DRIVER,NaN,2318330.0,NaN,FORD,F150,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2430973,cf7c14be28924d4de48bd41c392260cddc55c3c327f3bb...,2026-11-10T21:38:00.000,2,DRIVER,NaN,2318333.0,NaN,HYUNDAI,ELANTRA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2431010,bff635fd296b0e455677247d61576185908fe953963596...,2026-11-10T21:30:00.000,1,DRIVER,NaN,2318345.0,NaN,HYUNDAI,ACCENT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2431011,bff635fd296b0e455677247d61576185908fe953963596...,2026-11-10T21:30:00.000,2,PARKED,NaN,2318346.0,NaN,LAND ROVER,DISCOVERY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2431027,c779209213bb157f8fb1e841b75db10ea8e5086c7c905d...,2026-11-10T21:00:00.000,1,DRIVER,NaN,2318361.0,NaN,UNKNOWN,OTHER (EXPLAIN IN NARRATIVE),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:

df["crash_date"] = pd.to_datetime(
    df["crash_date"],
    errors="coerce"
)

print("Records by year:")
print(df["crash_date"].dt.year.value_counts().sort_index())

print("\nEarliest crash:", df["crash_date"].min())
print("Latest crash:", df["crash_date"].max())
print("Invalid dates:", df["crash_date"].isna().sum())


Records by year:
crash_date
2025    221863
2026    196614
Name: count, dtype: int64

Earliest crash: 2025-01-01 00:00:00
Latest crash: 2026-11-10 21:38:00
Invalid dates: 0


In [11]:

print("Dataset shape:", df.shape)

print("\nMissing values by column:")
print(df.isnull().sum().sort_values(ascending=False).to_string())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDuplicate CRASH_UNIT_ID:",
      df["crash_unit_id"].duplicated().sum())

print("\nDuplicate VEHICLE_ID (non-missing):",
      df.loc[
          df["vehicle_id"].notna(),
          "vehicle_id"
      ].duplicated().sum())

print("\nData types:")
print(df.dtypes.to_string())


Dataset shape: (418477, 71)

Missing values by column:
exceed_speed_limit_i        418476
hazmat_report_no            418475
mcs_report_no               418475
trailer2_length             418473
hazmat_name                 418470
wide_load_i                 418448
trailer2_width              418427
hazmat_placards_i           418417
un_no                       418321
idot_permit_no              418302
trailer1_length             418288
ilcc_no                     418270
hazmat_class                418267
total_vehicle_length        418236
trailer1_width              418164
ccmc_no                     418150
fire_i                      418131
axle_cnt                    418010
usdot_no                    416776
gvwr                        416698
commercial_src              416496
mcs_out_of_service_i        415784
load_type                   415776
hazmat_out_of_service_i     415775
mcs_report_i                415739
mcs_vio_cause_crash_i       415738
hazmat_report_i             415729


In [12]:

print("UNIT TYPES:")
print(df["unit_type"].value_counts(dropna=False))

print("\nMISSING VEHICLE IDs BY UNIT TYPE:")

result = pd.crosstab(
    df["unit_type"].fillna("MISSING"),
    df["vehicle_id"].isna(),
    normalize="index"
) * 100

result.columns = [
    "Vehicle ID Present (%)",
    "Vehicle ID Missing (%)"
]

print(result.round(2))


UNIT TYPES:
unit_type
DRIVER                 349778
PARKED                  54754
PEDESTRIAN               6028
BICYCLE                  5132
DRIVERLESS               1846
NON-MOTOR VEHICLE         531
NaN                       256
DISABLED VEHICLE           92
NON-CONTACT VEHICLE        58
EQUESTRIAN                  2
Name: count, dtype: int64

MISSING VEHICLE IDs BY UNIT TYPE:
                     Vehicle ID Present (%)  Vehicle ID Missing (%)
unit_type                                                          
BICYCLE                                0.00                  100.00
DISABLED VEHICLE                      97.83                    2.17
DRIVER                                99.99                    0.01
DRIVERLESS                            99.95                    0.05
EQUESTRIAN                             0.00                  100.00
MISSING                                0.00                  100.00
NON-CONTACT VEHICLE                   43.10                   56.90
NON-M

In [13]:

# Create a cleaning copy
df_clean = df.copy()

# Standardize column names
df_clean.columns = df_clean.columns.str.upper()

print("Column names standardized!")
print(df_clean.columns.tolist())


Column names standardized!
['CRASH_UNIT_ID', 'CRASH_RECORD_ID', 'CRASH_DATE', 'UNIT_NO', 'UNIT_TYPE', 'NUM_PASSENGERS', 'VEHICLE_ID', 'CMRC_VEH_I', 'MAKE', 'MODEL', 'LIC_PLATE_STATE', 'VEHICLE_YEAR', 'VEHICLE_DEFECT', 'VEHICLE_TYPE', 'VEHICLE_USE', 'TRAVEL_DIRECTION', 'MANEUVER', 'TOWED_I', 'FIRE_I', 'OCCUPANT_CNT', 'EXCEED_SPEED_LIMIT_I', 'TOWED_BY', 'TOWED_TO', 'AREA_00_I', 'AREA_01_I', 'AREA_02_I', 'AREA_03_I', 'AREA_04_I', 'AREA_05_I', 'AREA_06_I', 'AREA_07_I', 'AREA_08_I', 'AREA_09_I', 'AREA_10_I', 'AREA_11_I', 'AREA_12_I', 'AREA_99_I', 'FIRST_CONTACT_POINT', 'CMV_ID', 'USDOT_NO', 'CCMC_NO', 'ILCC_NO', 'COMMERCIAL_SRC', 'GVWR', 'CARRIER_NAME', 'CARRIER_STATE', 'CARRIER_CITY', 'HAZMAT_PLACARDS_I', 'HAZMAT_NAME', 'UN_NO', 'HAZMAT_PRESENT_I', 'HAZMAT_REPORT_I', 'HAZMAT_REPORT_NO', 'MCS_REPORT_I', 'MCS_REPORT_NO', 'HAZMAT_VIO_CAUSE_CRASH_I', 'MCS_VIO_CAUSE_CRASH_I', 'IDOT_PERMIT_NO', 'WIDE_LOAD_I', 'TRAILER1_WIDTH', 'TRAILER2_WIDTH', 'TRAILER1_LENGTH', 'TRAILER2_LENGTH', 'TOTAL_VEHICL

In [14]:

print("UNIT TYPES:")
print(df_clean["UNIT_TYPE"].value_counts(dropna=False))

print("\nMISSING VEHICLE IDs BY UNIT TYPE:")

result = pd.crosstab(
    df_clean["UNIT_TYPE"].fillna("MISSING"),
    df_clean["VEHICLE_ID"].isna(),
    normalize="index"
) * 100

result.columns = [
    "Vehicle ID Present (%)",
    "Vehicle ID Missing (%)"
]

print(result.round(2))


UNIT TYPES:
UNIT_TYPE
DRIVER                 349778
PARKED                  54754
PEDESTRIAN               6028
BICYCLE                  5132
DRIVERLESS               1846
NON-MOTOR VEHICLE         531
NaN                       256
DISABLED VEHICLE           92
NON-CONTACT VEHICLE        58
EQUESTRIAN                  2
Name: count, dtype: int64

MISSING VEHICLE IDs BY UNIT TYPE:
                     Vehicle ID Present (%)  Vehicle ID Missing (%)
UNIT_TYPE                                                          
BICYCLE                                0.00                  100.00
DISABLED VEHICLE                      97.83                    2.17
DRIVER                                99.99                    0.01
DRIVERLESS                            99.95                    0.05
EQUESTRIAN                             0.00                  100.00
MISSING                                0.00                  100.00
NON-CONTACT VEHICLE                   43.10                   56.90
NON-M

In [15]:

print("VEHICLE YEAR SUMMARY:")
print(df_clean["VEHICLE_YEAR"].describe())

print("\nVehicle years before 1900:",
      (df_clean["VEHICLE_YEAR"] < 1900).sum())

print("\nVehicle years after 2027:",
      (df_clean["VEHICLE_YEAR"] > 2027).sum())

print("\nMost common vehicle years:")
print(df_clean["VEHICLE_YEAR"].value_counts(dropna=False).head(15))


VEHICLE YEAR SUMMARY:
count    350162.000000
mean       2018.546738
std         143.137242
min        1900.000000
25%        2012.000000
50%        2017.000000
75%        2021.000000
max        9999.000000
Name: VEHICLE_YEAR, dtype: float64

Vehicle years before 1900: 0

Vehicle years after 2027: 276

Most common vehicle years:
VEHICLE_YEAR
NaN       68315
2017.0    21137
2015.0    20938
2018.0    19846
2016.0    19620
2020.0    19452
2019.0    19334
2024.0    19244
2014.0    18913
2023.0    18526
2021.0    17493
2013.0    17138
2025.0    16810
2022.0    16366
2012.0    14757
Name: count, dtype: int64


In [16]:

print("Vehicle years greater than 2027:")

print(
    df_clean.loc[
        df_clean["VEHICLE_YEAR"] > 2027,
        "VEHICLE_YEAR"
    ].value_counts().sort_index()
)

print("\nVehicle years equal to 1900:")
print((df_clean["VEHICLE_YEAR"] == 1900).sum())


Vehicle years greater than 2027:
VEHICLE_YEAR
2028.0     10
2029.0      4
2030.0      1
2031.0      1
2033.0      5
         ... 
7247.0      1
8147.0      1
9207.0      1
9900.0      1
9999.0    108
Name: count, Length: 84, dtype: int64

Vehicle years equal to 1900:
130


In [17]:

import numpy as np

# Identify invalid or placeholder vehicle years
invalid_years = (
    (df_clean["VEHICLE_YEAR"] == 1900) |
    (df_clean["VEHICLE_YEAR"] > 2027)
)

print("Invalid/placeholder years found:", invalid_years.sum())

# Replace with missing values
df_clean.loc[invalid_years, "VEHICLE_YEAR"] = np.nan

# Verify cleaning
print("\nRemaining years after 2027:",
      (df_clean["VEHICLE_YEAR"] > 2027).sum())

print("Remaining years equal to 1900:",
      (df_clean["VEHICLE_YEAR"] == 1900).sum())

print("Total missing vehicle years:",
      df_clean["VEHICLE_YEAR"].isna().sum())

print("\nDataset shape:", df_clean.shape)


Invalid/placeholder years found: 406

Remaining years after 2027: 0
Remaining years equal to 1900: 0
Total missing vehicle years: 68721

Dataset shape: (418477, 71)


In [18]:

columns_to_check = [
    "MAKE",
    "MODEL",
    "VEHICLE_TYPE",
    "VEHICLE_USE",
    "VEHICLE_DEFECT",
    "TRAVEL_DIRECTION",
    "MANEUVER",
    "LIC_PLATE_STATE"
]

for col in columns_to_check:
    print(f"\n===== {col} =====")
    print("Unique values:", df_clean[col].nunique())
    print(df_clean[col].value_counts(dropna=False).head(15))



===== MAKE =====
Unique values: 618
MAKE
TOYOTA                              51196
CHEVROLET                           42057
FORD                                40816
UNKNOWN                             37372
HONDA                               33224
NISSAN                              28820
JEEP                                19225
HYUNDAI                             16811
DODGE                               13163
KIA                                 12340
NaN                                 12040
VOLKSWAGEN                           8616
GENERAL MOTORS CORPORATION (GMC)     7421
LEXUS                                6934
MAZDA                                6715
Name: count, dtype: int64

===== MODEL =====
Unique values: 1659
MODEL
OTHER (EXPLAIN IN NARRATIVE)    64826
CAMRY                           12296
NaN                             12040
COROLLA                          9219
RAV4                             8321
MOTORIZED                        8161
CIVIC                        

In [19]:

# Standardize text formatting across categorical columns
categorical_cols = df_clean.select_dtypes(
    include=["object", "string"]
).columns.difference(["CRASH_RECORD_ID"])

for col in categorical_cols:
    df_clean[col] = df_clean[col].astype("string").str.strip().str.upper()
    df_clean[col] = df_clean[col].replace("", pd.NA)

# Clean license plate state placeholder
df_clean["LIC_PLATE_STATE"] = (
    df_clean["LIC_PLATE_STATE"].replace("XX", pd.NA)
)

# Verify
print("Remaining XX values:",
      (df_clean["LIC_PLATE_STATE"] == "XX").sum())

print("\nMissing license plate states:",
      df_clean["LIC_PLATE_STATE"].isna().sum())

print("\nDataset shape:", df_clean.shape)


Remaining XX values: 0

Missing license plate states: 51324

Dataset shape: (418477, 71)


In [20]:

print("FINAL VALIDATION")

print("\nDataset shape:", df_clean.shape)

print("\nDuplicate rows:",
      df_clean.duplicated().sum())

print("Duplicate CRASH_UNIT_ID:",
      df_clean["CRASH_UNIT_ID"].duplicated().sum())

print("Duplicate non-missing VEHICLE_ID:",
      df_clean.loc[
          df_clean["VEHICLE_ID"].notna(),
          "VEHICLE_ID"
      ].duplicated().sum())

print("\nMissing CRASH_RECORD_ID:",
      df_clean["CRASH_RECORD_ID"].isna().sum())

print("Missing CRASH_DATE:",
      df_clean["CRASH_DATE"].isna().sum())

print("\nRecords outside 2025–2026:",
      (~df_clean["CRASH_DATE"].dt.year.isin([2025, 2026])).sum())

print("\nRecords by year:")
print(df_clean["CRASH_DATE"].dt.year.value_counts().sort_index())

print("\nInvalid vehicle years:",
      (
          (df_clean["VEHICLE_YEAR"] == 1900) |
          (df_clean["VEHICLE_YEAR"] > 2027)
      ).sum())


FINAL VALIDATION

Dataset shape: (418477, 71)

Duplicate rows: 0
Duplicate CRASH_UNIT_ID: 0
Duplicate non-missing VEHICLE_ID: 0

Missing CRASH_RECORD_ID: 0
Missing CRASH_DATE: 0

Records outside 2025–2026: 0

Records by year:
CRASH_DATE
2025    221863
2026    196614
Name: count, dtype: int64

Invalid vehicle years: 0


In [21]:

from google.colab import files

# Save cleaned dataset
output_file = "Traffic_Crashes_Vehicles_Cleaned_2025_2026.csv"

df_clean.to_csv(output_file, index=False)

print("Cleaned dataset saved!")
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

# Download to your computer
files.download(output_file)


Cleaned dataset saved!
Rows: 418477
Columns: 71


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>